# Level 1 — Classic CNNs (VGG-16, ResNet-18 / 50)

**목표**: VGG 와 ResNet 을 직접 구현하여 Set A 위에서 Multi-task 로 학습하고, **Skip Connection** 이 깊은 네트워크의 수렴에 미치는 영향을 분석합니다.

**금지 사항**: `torchvision.models.*`, `timm`. 단, 위 라이브러리의 코드를 *참고하여 직접 타이핑* 하는 것은 허용됩니다.

본 노트북의 산출물:
1. 학습된 체크포인트 (`checkpoints/level1_*.pth`).
2. VGG-16, ResNet-18, ResNet-50 의 `Avg-Macro-F1` 및 속성별 Macro-F1 표.
3. VGG (skip 없음) vs ResNet (skip 있음) 의 학습 손실 곡선 비교.

In [1]:
import os
import sys

repo_url  = "https://github.com/Seongha-parkk/2026-HYU-AUE8088-PA2.git"
repo_name = "2026-HYU-AUE8088-PA2"

if not os.path.exists(f"/content/{repo_name}"):
    !git clone {repo_url}
else:
    # 런타임이 살아있고 이미 클론된 경우 최신 코드로 업데이트
    !git -C /content/{repo_name} pull

%cd /content/{repo_name}

Cloning into '2026-HYU-AUE8088-PA2'...
remote: Enumerating objects: 165, done.
remote: Counting objects: 100% (165/165), done.
remote: Compressing objects: 100% (117/117), done.
remote: Total 165 (delta 90), reused 120 (delta 45), pack-reused 0 (from 0)
Receiving objects: 100% (165/165), 976.71 KiB | 3.77 MiB/s, done.
Resolving deltas: 100% (90/90), done.
/content/2026-HYU-AUE8088-PA2


In [5]:
from google.colab import drive
import os

drive.mount('/drive')

DRIVE_CKPT = "/drive/MyDrive/aue8088-pa2/checkpoints"
os.makedirs(DRIVE_CKPT, exist_ok=True)

LOCAL_CKPT = os.path.abspath("../checkpoints")
if os.path.islink(LOCAL_CKPT):
    print(f"symlink 이미 존재: {LOCAL_CKPT} → {os.readlink(LOCAL_CKPT)}")
elif os.path.isdir(LOCAL_CKPT):
    import shutil
    for f in os.listdir(LOCAL_CKPT):
        shutil.move(os.path.join(LOCAL_CKPT, f), DRIVE_CKPT)
    shutil.rmtree(LOCAL_CKPT)
    os.symlink(DRIVE_CKPT, LOCAL_CKPT)
    print(f"기존 파일 Drive 이전 후 symlink 생성: {LOCAL_CKPT} → {DRIVE_CKPT}")
else:
    os.symlink(DRIVE_CKPT, LOCAL_CKPT)
    print(f"symlink 생성: {LOCAL_CKPT} → {DRIVE_CKPT}")

Mounted at /drive
symlink 생성: /content/checkpoints → /drive/MyDrive/aue8088-pa2/checkpoints


In [6]:
%load_ext autoreload
%autoreload 2

In [ ]:
!pip install -q -r requirements.txt

import torch
from torch.utils.data import DataLoader

from src.utils.seed import set_seed, seed_worker
from src.utils.transforms import train_transform, eval_transform
from src.utils.trainer import MultiTaskTrainer, TrainConfig
from src.utils.wandb_logger import WandbLogger
from src.utils.metrics import collect_predictions, confusion_matrices, CLASS_NAMES
from src.datasets.bdd_attr import BDDAttrDataset, ATTRIBUTES
from src.models.resnet import resnet18, resnet50
from src.models.vgg import VGG16

SEED = 42
set_seed(SEED, deterministic=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}  |  SEED={SEED}")

### Weights & Biases 설정

학습 곡선과 속성별 Macro-F1 을 자동으로 로깅합니다. 사용하지 않으려면 `WANDB_PROJECT = None` 으로 두세요 — 로깅 호출은 자동으로 no-op 이 됩니다.

최초 1회만:
```python
import wandb; wandb.login()   # API key 입력
```

In [8]:
import wandb; wandb.login()

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: qkrtjdgk16 (qkrtjdgk16-hanyang-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [10]:
WANDB_PROJECT = "aue8088-pa2"
WANDB_TAGS    = ["level1"]
# 환경변수로도 끌 수 있음:  os.environ["WANDB_DISABLED"] = "true"

In [11]:
DATA_ROOT = "../data/set_a"
BATCH = 64

# ../data/set_a 가 없으면 zip 을 받아 상위 폴더에 압축 해제 → ../data/set_a, ../data/set_b 생성.
import os, sys, zipfile, subprocess

GDRIVE_FILE_ID = "1L7YC70QlO87aIbE5lbtQ94HUINJijBKK"
ZIP_PATH   = "../aue8088_pa2_data.zip"
EXTRACT_TO = ".."   # zip 내부 최상위가 data/ 이므로 상위 폴더에 풀면 ../data/... 가 됨

if not os.path.isdir(DATA_ROOT):
    try:
        import gdown
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
        import gdown

    if not os.path.exists(ZIP_PATH):
        print("데이터셋 zip 다운로드 중...")
        gdown.download(id=GDRIVE_FILE_ID, output=ZIP_PATH, quiet=False)

    print("압축 해제 중...")
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extractall(EXTRACT_TO)
    print(f"완료 → {DATA_ROOT}")
else:
    print(f"데이터셋이 이미 존재합니다 → {DATA_ROOT}")

train_ds = BDDAttrDataset(DATA_ROOT, split="train", transform=train_transform())
val_ds   = BDDAttrDataset(DATA_ROOT, split="val",   transform=eval_transform())

g = torch.Generator(); g.manual_seed(SEED)
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=2, worker_init_fn=seed_worker, generator=g, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)

for a in ATTRIBUTES:
    print(f"{a:10s} 클래스별 샘플 수 (train) = {train_ds.class_counts(a).tolist()}")

데이터셋 zip 다운로드 중...


Downloading...
From (original): https://drive.google.com/uc?id=1L7YC70QlO87aIbE5lbtQ94HUINJijBKK
From (redirected): https://drive.google.com/uc?id=1L7YC70QlO87aIbE5lbtQ94HUINJijBKK&confirm=t&uuid=1b498aec-0dc3-4d3c-9f03-c26c87ff9710
To: /content/aue8088_pa2_data.zip
100%|██████████| 243M/243M [00:03<00:00, 78.5MB/s] 


압축 해제 중...
완료 → ../data/set_a
weather    클래스별 샘플 수 (train) = [3100, 800, 400, 200, 0, 500]
scene      클래스별 샘플 수 (train) = [3052, 1386, 562]
timeofday  클래스별 샘플 수 (train) = [2479, 2129, 392]


In [ ]:
from torch import nn

def train_one(model_fn, name, epochs=20):
    """단일 모델을 학습하고 체크포인트와 wandb 산출물을 저장한다."""
    ckpt_path = f"../checkpoints/level1_{name}.pth"

    if os.path.exists(ckpt_path):
        m = model_fn().to(device)
        m.load_state_dict(torch.load(ckpt_path, map_location=device)["state_dict"])
        m.eval()
        print(f"{name} ← {ckpt_path}")
        return m, {}

    set_seed(SEED, deterministic=True)
    model = model_fn().to(device)
    optim = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=5e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=epochs)
    losses = {a: nn.CrossEntropyLoss() for a in ATTRIBUTES}
    cfg = TrainConfig(epochs=epochs)

    logger = WandbLogger(
        project=WANDB_PROJECT,
        run_name=f"level1-{name}",
        config={
            "backbone": name, "epochs": epochs, "batch": BATCH,
            "lr": 3e-4, "weight_decay": 5e-4, "seed": SEED,
            "loss_weights": cfg.loss_weights,
        },
        tags=WANDB_TAGS + [name],
    )

    trainer = MultiTaskTrainer(model, optim, sched, losses, device, cfg, logger=logger)
    history = trainer.fit(train_loader, val_loader)

    val_pred, _, val_tgt, _ = collect_predictions(model, val_loader, device)
    cms = confusion_matrices(val_pred, val_tgt)
    for a in ATTRIBUTES:
        logger.log_confusion_matrix(f"final/cm_{a}", cms[a], CLASS_NAMES[a])
    logger.finish()

    os.makedirs("../checkpoints", exist_ok=True)
    torch.save({"state_dict": model.state_dict(), "history": history}, ckpt_path)
    return model, history

vgg_model, vgg_hist = train_one(VGG16,    "vgg16",    epochs=30)
r18_model, r18_hist = train_one(resnet18, "resnet18", epochs=30)
r50_model, r50_hist = train_one(resnet50, "resnet50", epochs=30)

In [ ]:
# Uncertainty Weighting 실험 (Kendall et al., CVPR 2018)
# Reference: https://arxiv.org/abs/1705.07115
from torch import nn
from src.losses.imbalanced import UncertaintyWeighting

def train_one_uw(model_fn, name, epochs=30):
    """Uncertainty Weighting 으로 학습. σ 값이 wandb 에 자동 로깅된다."""
    ckpt_path = f"../checkpoints/level1_{name}_uw.pth"

    if os.path.exists(ckpt_path):
        m = model_fn().to(device)
        m.load_state_dict(torch.load(ckpt_path, map_location=device)["state_dict"])
        m.eval()
        print(f"{name}_uw ← {ckpt_path}")
        return m, {}, None

    set_seed(SEED, deterministic=True)
    model = model_fn().to(device)
    uw = UncertaintyWeighting(list(ATTRIBUTES)).to(device)

    optim = torch.optim.AdamW(
        list(model.parameters()) + list(uw.parameters()),
        lr=3e-4, weight_decay=5e-4,
    )
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=epochs)
    losses = {a: nn.CrossEntropyLoss() for a in ATTRIBUTES}
    cfg = TrainConfig(epochs=epochs)

    logger = WandbLogger(
        project=WANDB_PROJECT,
        run_name=f"level1-{name}-uw",
        config={
            "backbone": name, "epochs": epochs, "batch": BATCH,
            "lr": 3e-4, "weight_decay": 5e-4, "seed": SEED,
            "loss_weighting": "uncertainty",
        },
        tags=WANDB_TAGS + [name, "uncertainty-weighting"],
    )

    trainer = MultiTaskTrainer(model, optim, sched, losses, device, cfg, logger=logger, uw=uw)
    history = trainer.fit(train_loader, val_loader)

    print(f"\n[{name}] 최종 σ 값 (클수록 어려운 task):")
    for attr, sigma in uw.sigmas().items():
        print(f"  σ_{attr:<10s} = {sigma:.4f}  (가중치 ≈ {1 / (2 * sigma**2):.4f})")

    val_pred, _, val_tgt, _ = collect_predictions(model, val_loader, device)
    cms = confusion_matrices(val_pred, val_tgt)
    for a in ATTRIBUTES:
        logger.log_confusion_matrix(f"final/cm_{a}", cms[a], CLASS_NAMES[a])
    logger.finish()

    os.makedirs("../checkpoints", exist_ok=True)
    torch.save({"state_dict": model.state_dict(), "history": history}, ckpt_path)
    return model, history, uw

vgg_uw,  vgg_hist_uw,  vgg_uw_module  = train_one_uw(VGG16,    "vgg16",    epochs=30)
r18_uw,  r18_hist_uw,  r18_uw_module  = train_one_uw(resnet18, "resnet18", epochs=30)
r50_uw,  r50_hist_uw,  r50_uw_module  = train_one_uw(resnet50, "resnet50", epochs=30)

## 분석 (리포트 필수 포함 항목)

1. **수렴 비교**: VGG-16 과 ResNet-18 의 학습 손실 곡선을 함께 그리세요. Skip connection 이 없는 깊은 네트워크가 정체되는 현상이 관찰되는지, 그 원인은 무엇인지 서술하세요.
2. **속성별 MF1 표**: 각 백본에 대해 Weather / Scene / Time of Day 의 Macro-F1 을 표로 정리하세요. 어느 속성이 가장 어려운지, 깊이 변화에 따라 그 양상이 어떻게 바뀌는지 분석하세요.
3. **Loss 가중치 민감도**: ResNet-18 에 대해 최소 두 가지 비자명한 가중치 설정 (예: `(1, 1, 1)` vs `(2, 1, 1)`) 으로 재학습하고 Avg-MF1 변화를 보고하세요.

wandb 를 활성화한 경우 같은 프로젝트의 Run 들을 비교하면 학습 곡선·속성별 MF1·confusion matrix 가 한눈에 정리됩니다.